In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/used_device_data.csv")
print(df.shape)
print(df.head())

(3454, 15)
  device_brand       os  screen_size   4g   5g  rear_camera_mp  \
0        Honor  Android        14.50  yes   no            13.0   
1        Honor  Android        17.30  yes  yes            13.0   
2        Honor  Android        16.69  yes  yes            13.0   
3        Honor  Android        25.50  yes  yes            13.0   
4        Honor  Android        15.32  yes   no            13.0   

   front_camera_mp  internal_memory  ram  battery  weight  release_year  \
0              5.0             64.0  3.0   3020.0   146.0          2020   
1             16.0            128.0  8.0   4300.0   213.0          2020   
2              8.0            128.0  8.0   4200.0   213.0          2020   
3              8.0             64.0  6.0   7250.0   480.0          2020   
4              8.0             64.0  3.0   5000.0   185.0          2020   

   days_used  normalized_used_price  normalized_new_price  
0        127               4.307572              4.715100  
1        325         

In [ ]:
#create device age
current_year = 2026
df["device_age"] = current_year - df["release_year"]
print(df["device_age"].head())

0    6
1    6
2    6
3    6
4    6
Name: device_age, dtype: int64


In [ ]:
df = df.drop(
    columns=[
        "release_year"
    ]
)

In [ ]:
df["4g"] = df["4g"].map({"yes": 1, "no": 0})
df["5g"] = df["5g"].map({"yes": 1, "no": 0})

In [ ]:
print(df.head())

  device_brand       os  screen_size  4g  5g  rear_camera_mp  front_camera_mp  \
0        Honor  Android        14.50   1   0            13.0              5.0   
1        Honor  Android        17.30   1   1            13.0             16.0   
2        Honor  Android        16.69   1   1            13.0              8.0   
3        Honor  Android        25.50   1   1            13.0              8.0   
4        Honor  Android        15.32   1   0            13.0              8.0   

   internal_memory  ram  battery  weight  days_used  normalized_used_price  \
0             64.0  3.0   3020.0   146.0        127               4.307572   
1            128.0  8.0   4300.0   213.0        325               5.162097   
2            128.0  8.0   4200.0   213.0        162               5.111084   
3             64.0  6.0   7250.0   480.0        345               5.135387   
4             64.0  3.0   5000.0   185.0        293               4.389995   

   normalized_new_price  device_age  
0     

In [ ]:
# One Hot encode brand and Os
df = pd.get_dummies(
    df,
    columns=["device_brand", "os"],
    drop_first=True
)

In [ ]:
print(df.head())

   screen_size  4g  5g  rear_camera_mp  front_camera_mp  internal_memory  ram  \
0        14.50   1   0            13.0              5.0             64.0  3.0   
1        17.30   1   1            13.0             16.0            128.0  8.0   
2        16.69   1   1            13.0              8.0            128.0  8.0   
3        25.50   1   1            13.0              8.0             64.0  6.0   
4        15.32   1   0            13.0              8.0             64.0  3.0   

   battery  weight  days_used  ...  device_brand_Samsung  device_brand_Sony  \
0   3020.0   146.0        127  ...                 False              False   
1   4300.0   213.0        325  ...                 False              False   
2   4200.0   213.0        162  ...                 False              False   
3   7250.0   480.0        345  ...                 False              False   
4   5000.0   185.0        293  ...                 False              False   

   device_brand_Spice  device_brand_Vi

In [ ]:
# Features & Target
X = df.drop("normalized_used_price", axis=1)
y = df["normalized_used_price"]

In [ ]:
print(X.shape)

(3454, 48)


In [ ]:
# Train test split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=15, n_estimators=300, n_jobs=-1,
                      random_state=42)

In [ ]:
# Prediction
y_pred = rf.predict(X_test)

In [ ]:
# Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

MAE : 0.17
RMSE: 0.22
R2 Score: 0.8568


In [ ]:
# Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(15))

                 Feature  Importance
10             new_price    0.629810
0            screen_size    0.121439
5        internal_memory    0.050741
3         rear_camera_mp    0.037396
4        front_camera_mp    0.035982
9              days_used    0.034423
8                 weight    0.030386
7                battery    0.021683
11            device_age    0.009151
6                    ram    0.005553
43   device_brand_Xiaomi    0.004423
38  device_brand_Samsung    0.002255
35   device_brand_Others    0.001499
22   device_brand_Huawei    0.001473
2                     5g    0.001194


In [ ]:
sample = X_test.iloc[[0]]

prediction = rf.predict(sample)

print("Predicted Used Price: $", round(prediction[0], 2))
print("Actual Used Price   : $", round(y_test.iloc[0], 2))

Predicted Used Price: $ 56.58
Actual Used Price   : $ 53.21
